# Understanding CRDS and How to Select Calibration Reference Files

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.1".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.
    

## Introduction
This notebook will help you understand how the Calibration Reference Data System ([CRDS](https://roman-crds.stsci.edu/static/users_guide/overview.html)) works, how the Roman Calibration Pipeline interacts with this subsystem to get the best reference files for your data, and how to retrieve these files directly from `CRDS` 


### Local Run Settings

If you want to run the notebook in your local machine, refer to the information in [local installation](../../markdown/local-run.md) instructions before proceeding with the notebook. The instructions provide important information about setting up your environment and installing dependencies.

## Imports
Libraries used:
- *copy* for making copies of Python objects
- *crds* for access to calibration reference files
- *numpy* for array manipulation
- *roman_datamodels* for opening Roman WFI ASDF files
- *asdf* for opening Roman WFI ASDF files
- *os* for operating system functions

In [ ]:
import os
import copy

import numpy as np
import roman_datamodels as rdm
#import s3fs

### The Calibration Reference Data System (CRDS)

The Roman Calibration Pipeline (`RomanCal`), uses calibration reference and parameter files from the [CRDS](https://roman-crds.stsci.edu/static/users_guide/overview.html). These reference files, developed and validated by STScI’s Science Operations Center, are continually updated as new WFI data become available. CRDS assigns the most appropriate reference file for each calibration step using metadata keywords and file-specific matching criteria. To use the best-available reference files for an observation, no action is needed as `RomanCal` will query for the best reference files for each calibration step in the pipeline.

To do this independently we will use the **[`crds`](https://roman-crds.stsci.edu/static/users_guide/index.html)** Python application programming interface (API); however, the [**CRDS** webserver](https://roman-crds.stsci.edu) can also be accessed to browse calibration reference files in a tabular interface. Note that there are multiple CRDS servers, though most users will interact with the Operations (OPS) instance. Please be sure to navigate to the correct webserver for the instance in which you are interested.

For more details, see the [RDox page on CRDS for Roman WFI](https://roman-docs.stsci.edu/data-handbook-home/accessing-wfi-data/crds-for-reference-files) and the [CRDS documentation](https://jwst-crds.stsci.edu/static/users_guide/web_site_use.html).

In [ ]:
import crds

Check your environment. If running the notebook in Nexus, these CRDS environment variables would be already configured. If running locally and not currently set up, don't worry, we will select the CRDS context later in the notebook.

In [ ]:
print("CRDS_SERVER_URL:", os.environ.get('CRDS_SERVER_URL'))
print("CRDS_CONTEXT:", os.environ.get('CRDS_CONTEXT'))
print("CRDS_PATH:", os.environ.get('CRDS_PATH'))

### Retrieving Reference Files

As you run the exposure or mosaic pipeline, the most up-to-date reference files will be automatically selected for each step. However, if you would like to use a specific reference file, these can be retrieved through the `crds` Python API and stored localy to be used later in a particular calibraton step  (more on that later). Let's begin with how to access reference files from CRDS.

First, let's begin with how to access reference files from CRDS. For that we will use the `crds.getrecommendations()` function. This function returns a dictionary with the basenames of the reference files that match the criteria that you supply. In this case, the files are not downloaded.

In this function, the selection criteria are specified in a dictionary of key-value pairs, where eacb Roman WFI metadata keyword in the dictionary is all-caps and always begins with "ROMAN.META.". The remaining parts of the string correspond to the metadata keyword locations in the science data file schema. To lear more about Science Products Schemas refer to the [Roman Atribute Dictionary](https://rad.readthedocs.io/en/latest/schemas.html).

Different reference file types require different combinations of science metadata to match to the reference files. In general, all reference file types will require the instrument name ("INSTRUMENT.NAME") and start time ("EXPOSURE.START_TIME"). Most file types require the detector name ("INSTRUMENT.DETECTOR"), and some file types require the exposure type ("EXPOSURE.TYPE") or optical element ("INSTRUMENT.OPTICAL_ELEMENT").

For example, for the mask and flat files, the required keywords are:
- mask
    - ROMAN.META.INSTRUMENT.NAME
    - ROMAN.META.INSTRUMENT.DETECTOR
    - ROMAN.META.EXPOSURE.START_TIME
- flat
    - ROMAN.META.INSTRUMENT.NAME
    - ROMAN.META.INSTRUMENT.DETECTOR
    - ROMAN.META.INSTRUMENT.OPTICAL_ELEMENT
    - ROMAN.META.EXPOSURE.START_TIME

These keywords may be combined into a single dictionary to find multiple reference file types using `crds.getreferences()`. This function returns full local paths and downloads or caches if missing. For example, if you would like to find the name of the dark and flat reference files used by the pipeline, and download them, you could run the following example:

In [ ]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
        'ROMAN.META.INSTRUMENT.OPTICAL_ELEMENT': 'F158',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getrecommendations(meta, reftypes=['mask', 'flat'], observatory='roman')

The `ref_files` variable now contains a dictionary for each of the reference file types you requested (MASK and FLAT). These are the reference files that correspond to a science observation taken at midnight UTC on January 1, 2026 in the WFI imaging mode with optical element F158 and detector WFI01. Let's take a look at the names of the files CRDS returned:

In [ ]:
ref_files

We can also use `crds.getreferences()` to accomplish the same thing; however, `getreferences()` goes one step further beyond `getrecommendations()` and will download the reference files if they are not already in your local cache. Using the same example as above:

In [ ]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
        'ROMAN.META.INSTRUMENT.OPTICAL_ELEMENT': 'F158',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getreferences(meta, reftypes=['mask', 'flat'], observatory='roman')

And once again we can examine the output of `ref_files`:

In [ ]:
ref_files

This time, `ref_files` contains the path to the file in the local cache (controlled by the `CRDS_PATH` environment variable) since we did not simply ask for the file name but also checked if the file was in our cache, and if it was not then we downloaded it.

### CRDS Mapping Files

CRDS organizes the reference files using mapping files: the PMAP file, the IMAP files, and tje RMAP files. The first and most important mapping file is the PMAP file (commonly referred to as the CRDS context). This file is set by the the `CRDS_CONTEXT` environment variable. It provides a way to identify to the full set of reference files that each step of the `RomanCal` pipeline will use at a given time. By using the reference files tied to the same PMAP, you ensure compatibility between reference files.

If not already set, we can select a particular context with the following command.

In [ ]:
os.environ['CRDS_CONTEXT']='roman_0055.pmap'

Let's check again our environment variables

In [ ]:
print(f"CRDS server location: {os.environ.get('CRDS_SERVER_URL')}")
print(f"CRDS context file: {os.environ.get('CRDS_CONTEXT')}")

We can see that the PMAP was correctly set.

The IMAP is the second tier of mapping files. It contains the full set of reference types applicable to a giver instrument and available for use by `RomanCal`. For WFI, a particular version of the IMAP provides a list of all the reference file types used by `RomanCal` to calibrate that instrument, along with the RMAP versions part of the selected context. 

The RMAP is the third tier of mapping files. It provides the full list of reference files active with a given context, along with the selection parameters for that particular reference file type, which matches them to science observations. An RMAP includes the versions of the reference file that is part of that context.

From the user point of view, only the PMAP or context file needs to be set. Note, however, that the context is also CRDS server-dependent and is set by the environment variable `CRDS_SERVER_URL`. 

Let's explore the set of maping associated with our selected context.

In [ ]:
maps = pmap.mapping_names()
print(maps)

For more information on the mapping rules, see the [readthedocs documentation](https://roman-crds.stsci.edu/static/users_guide/rmap_syntax.html). 

Now let's isolate the names of the MASK files in the associated RMAP.

In [ ]:
for m in maps:
    if 'mask' in m:
        mask_rmap = m 
        break
    else:
        pass

masks = crds.rmap.load_mapping(mask_rmap)

Now that we have loaded up the list of MASK files in the applicable RMAP, let's take a look at it using the `todict()` method to turn it into a dictionary we can more easily visualize:

In [ ]:
masks.todict()

As we can see, the `parkey` key in the RMAP tells us the matching criteria unique to this reference file type from the dictionary we created when we used `crds.getrecommendations()` and `crds.getreferences()`. Also notice that the `parameters` field tells us the column names for the `selections` part of the dictionary. In this case, we see they include:

    "ROMAN.META.INSTRUMENT.DETECTOR"
    "USEAFTER"
    "REFERENCE".     

The REFERENCE column is simply the name of the reference file that matches those critiera. The USEAFTER date is the date after which a file should be used for a science observation. 
CRDS will match the file with the closest **preceding** USEAFTER date to the science observation date (given by "ROMAN.META.EXPOSURE.START_TIME").

### Download a File by Name

If you know the specific reference file name and would like to download it directly from the CRDS on-premise server, you can call it via a command line option:

```
crds sync --files <filename> --output-dir=<pathname>
```

where `<filename>` is the name of the reference file (e.g. "roman_wfi_mask_0022.asdf") and `<pathname>` is the location where the file will be saved. **Note:** in the future, Roman reference files will also be available via an AWS S3 bucket, and these instructions will be updated to describe how to access them there. 

To run this in a Jupyter notebook, we write out the command as a string and pass it to the command line script code:

In [ ]:
cmd = f"crds.sync --files {ref_files['mask']}"
_ = crds.sync.SyncScript(cmd)()

In this case, nothing happened since we had already previously downloaded this file into our cache using `crds.getreferences()`. But if you know the file name of a reference file that you want to retrieve from CRDS without using the matching criteria, the cell above would download the file to your cache. Simply replace `{ref_files['mask']}` in the `cmd` string with the name of the ASDF file.


In [ ]:
mask = rdm.open(ref_files['mask'])
mask.info()

## What is Next 
* Understand and Explore the [Bad Pixels Mask](bad_pixels_mask_reffile.ipynb) reference file.
* Understand and Explore the [Dark](dark_reffile.ipynb) reference file.
* Understand and Explore the [Flat Field](flat_reffile.ipynb) reference file.

----

## Additional Resources
Additional information can be found at the following links:

??????

## About this Notebook
**Author:** R. Diaz, T. Desjardins.

**Updated On:** 2026-07-02

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>